## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


In [0]:
tf.enable_eager_execution()

### Collect Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd

In [0]:
data = pd.read_csv('/content/drive/My Drive/GL_AIML/Residency_6/internal_lab/prices.csv')


### Check all columns in the dataset

In [6]:
data.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Drop columns `date` and  `symbol`

In [7]:
df=data.drop('date',axis=1)
# df=data.drop('symbol',axis=1)
df.head()

,symbol,open,close,low,high,volume
0,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


In [8]:
df=df.drop('symbol',axis=1)
df.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [9]:
df1=df.iloc[:1000]
df1.shape

(1000, 5)

### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split

target = df1["volume"]
features = df1.drop(["volume"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(features,target, test_size = 0.3, random_state = 10)

#### Convert Training and Test Data to numpy float32 arrays


In [0]:
import numpy as np
X_train = X_train.astype('float32') 
X_train=np.array(X_train)
# X_train=np.array(X_train,dtype='float32')
X_test = X_test.astype('float32') 
X_test=np.array(X_test)
y_train = y_train.astype('float32') 
y_train=np.array(y_train)
y_test = y_test.astype('float32') 
y_test=np.array(y_test)


In [12]:
type(X_train)

numpy.ndarray

In [13]:
# X_train.dtype
#X_test.dtype
y_train.dtype
#print('datatype of y_test is', y_test.dtype)

dtype('float32')

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import Normalizer
# from sklearn import preprocessing
transformer = Normalizer()

n_X_train= transformer.transform(X_train)
n_X_test= transformer.transform(X_test)

In [0]:
# scaler=StandardScaler()
# scaled_data_X_train = scaler.fit_transform(X_train)
# scaled_data_X_test = scaler.fit_transform(X_test)


## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

In [81]:
W

<tf.Variable 'Weights_4:0' shape=(4, 1) dtype=float32_ref>

2.Define a function to calculate prediction

In [0]:
#We will use normalized data
#y = tf.add(tf.matmul(x,W),b,name='output')
def prediction(n_X_train,W,b):
  return tf.add(tf.matmul(n_X_train,W),b,name='output')  
# y = tf.add(tf.matmul(n_X_train,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [0]:
def loss_c(n_X_train,y_train,W,b):
  return tf.reduce_mean(tf.square(y_train - prediction(n_X_train,W,b)),name='Loss')
# loss = tf.reduce_mean(tf.square(y-y_train),name='Loss')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
# train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)
def training(n_X_train,y_train,W,b,learning_rate=0.03):
  with tf.GradientTape() as tape:
    tape.watch([W,b])
    loss = loss_c(n_X_train,y_train,W,b)
    dW,db=tape.gradient(loss,[W,b])
    W=W-learning_rate * dW
    b=b-learning_rate * db
    return W,b

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [22]:
#Lets start graph Execution
#sess = tf.Session()

# variables need to be initialized before we can use them or resetting all global variables
#sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
#training_epochs = 100

for epoch in range(100):
  W1,b1=training(n_X_train,y_train,W,b,0.03)
  if epoch%5==0:
    print('Training loss at step: ',epoch,' is ', loss_c(n_X_train,y_train,W1,b1))

Training loss at step:  0  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  5  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  10  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  15  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  20  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  25  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  30  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  35  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  40  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  45  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  50  is  tf.Tensor(302197860000000.0, shape=(), dtype=float32)
Training loss at step:  55  is  tf.Tensor(30219786000000

In [90]:
n_X_train.shape

(700, 4)

In [0]:
#for epoch in range(training_epochs):
            
    #Calculate train_op and loss, feed_dict is hyper parameter with x train and y_tarin and for run have multiple Options -
    # - We can test function no need to pass sess.run in that case Pass key value of test data as x_test and y_test -
    # for y_pridict sess.run([train_op])
#    _, train_loss = sess.run([train_op,loss],feed_dict={x:n_X_train, y_:y_train})
    
#    if epoch % 5 == 0:
#        print ('Training loss at step: ', epoch, ' is ', train_loss)

### Get the shapes and values of W and b

### Model Prediction on 1st Examples in Test Dataset

In [0]:
y_pred_1st=prediction(n_X_test[0:1],W1,b1)

In [27]:
print(y_pred_1st)

tf.Tensor([[687957.9]], shape=(1, 1), dtype=float32)


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [29]:
iris_df=pd.read_csv('/content/drive/My Drive/GL_AIML/Residency_6/internal_lab/Iris.csv')
iris_df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
iris_1 = iris_df.join(pd.get_dummies(iris_df["Species"]))
y = iris_1[["Iris-setosa", "Iris-versicolor", "Iris-virginica"]]

### Splitting the data into feature set and target set

In [31]:
y.head()

,Iris-setosa,Iris-versicolor,Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [0]:
# Create a separate dataframe consisting only of the features i.e independent attributes
X = iris_df.drop(labels= ["Species","Id"], axis = 1)

In [34]:

print('Number of examples: ', X.shape[0])
print('Number of features for each example: ', X.shape[1])
print('Shape of actual data: ', y.shape)

Number of examples:  150
Number of features for each example:  4
Shape of actual data:  (150, 3)


In [0]:
seed = 7
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=seed)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
#from keras.models import Sequential
#from keras.layers import Dense

#Initialize Sequential Graph (model)
model = tf.keras.Sequential()

#Normalize input data
model.add(tf.keras.layers.Dense(3, input_shape=(4,), activation='softmax'))

#Compile the model - add Loss and Gradient Descent optimizer
model.compile(optimizer='sgd', loss='categorical_crossentropy')

### Model Training 

In [38]:
model.fit(X_train, y_train, epochs=5)

W0721 13:31:01.730242 139772795123584 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/tensorflow/python/ops/math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Epoch 1/5
105/105 [==============================] - 0s 1ms/sample - loss: 2.5981
Epoch 2/5
105/105 [==============================] - 0s 125us/sample - loss: 1.7631
Epoch 3/5
105/105 [==============================] - 0s 106us/sample - loss: 1.4489
Epoch 4/5
105/105 [==============================] - 0s 89us/sample - loss: 1.3808
Epoch 5/5
105/105 [==============================] - 0s 123us/sample - loss: 1.3431


### Model Prediction

In [39]:
model.predict(X_test)

array([[0.2054698 , 0.39943367, 0.3950965 ],
       [0.17328022, 0.43080723, 0.39591253],
       [0.15876804, 0.42252037, 0.41871154],
       [0.19101533, 0.41537184, 0.39361286],
       [0.22362474, 0.3814242 , 0.3949511 ],
       [0.32804576, 0.31388488, 0.35806942],
       [0.28339192, 0.3547352 , 0.36187285],
       [0.24776934, 0.38067505, 0.3715556 ],
       [0.13645196, 0.46514392, 0.3984042 ],
       [0.23559724, 0.37597132, 0.38843143],
       [0.29158106, 0.33900595, 0.36941296],
       [0.17141631, 0.4168319 , 0.41175178],
       [0.08708118, 0.5123692 , 0.40054965],
       [0.2205718 , 0.3547565 , 0.42467174],
       [0.17808422, 0.41914132, 0.40277445],
       [0.19043243, 0.39801717, 0.41155037],
       [0.34919629, 0.3230799 , 0.3277239 ],
       [0.21217433, 0.40569422, 0.3821315 ],
       [0.11059922, 0.4849152 , 0.40448558],
       [0.18530369, 0.429938  , 0.38475826],
       [0.28392425, 0.35189602, 0.36417973],
       [0.27546105, 0.32465124, 0.39988768],
       [0.

### Save the Model

In [0]:
model.save('my_model.h5') 

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?

In [0]:
#Initialize Sequential Graph (model)
model = tf.keras.Sequential()

#Normalize input data
model.add(tf.keras.layers.Dense(2, input_shape=(4,), activation='softmax'))

#model.add(tf.keras.layers.Dense(3, input_shape=(4,), activation='softmax'))

#Add Dense layer for prediction - Keras declares weights and bias automatically
model.add(tf.keras.layers.Dense(3))

#Compile the model - add Loss and Gradient Descent optimizer
model.compile(optimizer='sgd', loss='categorical_crossentropy')

In [44]:
model.predict(X_test)

array([[-0.15292518,  0.06042479, -1.0235445 ],
       [-0.09684924,  0.1529516 , -1.0244352 ],
       [ 0.11580729,  0.5038406 , -1.0278128 ],
       [-0.0972285 ,  0.1523258 , -1.0244292 ],
       [-0.17936423,  0.01679963, -1.0231247 ],
       [ 0.04369618,  0.38485536, -1.0266675 ],
       [-0.1130994 ,  0.12613842, -1.0241772 ],
       [-0.11246677,  0.12718226, -1.0241872 ],
       [ 0.10733417,  0.48985973, -1.0276781 ],
       [-0.06993181,  0.1973661 , -1.0248628 ],
       [-0.14314398,  0.07656401, -1.0236999 ],
       [-0.04552908,  0.23763125, -1.0252503 ],
       [ 0.13179399,  0.5302191 , -1.0280666 ],
       [-0.16549534,  0.03968369, -1.0233449 ],
       [ 0.11110071,  0.49607465, -1.0277381 ],
       [-0.10136408,  0.14550197, -1.0243634 ],
       [-0.27440736, -0.14002404, -1.0216151 ],
       [-0.16762868,  0.03616362, -1.0233111 ],
       [ 0.12195224,  0.51398   , -1.0279104 ],
       [ 0.09652228,  0.47201982, -1.0275065 ],
       [-0.08782981,  0.1678339 , -1.024

In [45]:
scores = model.evaluate(X_test, y_test)
scores

45/45 [==============================] - 0s 738us/sample - loss: 10.0937


10.09365628560384